# Readme


This routine executes the auto-eventing algorithm on the dataset of choice. 

# Load modules

In [1]:
import sys
import logging
from datetime import datetime, timedelta

import pandas as pd

import itertools

import json
import numpy as np
from time import time
import os
from tqdm import tqdm
from copy import copy

In [2]:
source_path = '/Users/fvidal/Documents/TTY/fifa/'
sys.path.insert(0, source_path+'auto_eventing/')

from game_time import GameTime, _display_football_time
from tracking_routines import *
from set_inputs import *
from football_logic import possession_model, event_detection
from utils import decompress_pickle, compressed_pickle


# Initialize game and parameters

#### Paths to data

In [3]:
dataset = 'mwc_22'
provider = 'tracab5' 
    
output_path = source_path + f'Datasets/{dataset}/autoevents'
metadata_path = source_path + f'Datasets/{dataset}/IFES/'
tracking_raw_path = source_path + f'Datasets/{dataset}/tracking/{provider}/'

sys.path.insert(0, output_path)

today = datetime.today().strftime('%Y-%m-%d')
this_run = '/'+today+'_'+provider

save_data = True
verbose = True

#### Specify games to run auto-eventing on 

The list of games is fetched from the IFES Excel files

In [4]:
list_of_games = [str(s) for s in pd.read_excel(metadata_path+'Match IDs.xlsx')['Match ID'].tolist()]
print(list_of_games)

['128084', '132997', '132996', '132995', '133001', '132999', '132998', '133000', '133002', '133005', '133003', '133004', '133007', '133017', '133006', '133008', '133012', '133010', '133009', '133011', '133014', '133016', '133013', '133015', '133018', '133019', '133020', '133021', '133023', '133022', '133024', '133025', '133026', '133028', '133027', '133029', '133033', '133031', '133030', '133032', '133036', '133034', '133037', '133035', '133038', '133041', '133040', '133039', '128068', '128069', '128070', '128071', '128072', '128073', '128074', '128075', '128076', '128077', '128078', '128079', '128080', '128081', '128082', '128083']


Single game (e.g. the MWC22 final)

In [5]:
# list_of_games = ['128083']

### Set parameters


In [6]:
args = {'postprocess': True, 'compute_modes':True, 
                           'compute_pressures':True, 'compute_pitch_control':True}
params_base = get_params(provider, dataset, **args)

print(params_base)

{'provider': 'tracab5', 'dataset': 'mwc_22', 'fps': 25, 'verbose': True, 'set_event_logic': True, 'set_possession_logic': True, 'postprocess': True, 'compute_modes': True, 'compute_pressures': True, 'compute_pitch_control': True, 'pitch_measures': {'penspot_dim': 10.9728, 'smbx_dim': 5.4864, 'goal_dim': 7.3152, 'arc_dim': 9.144, 'penbox_len': 16.4592, 'penbox_wid': 40.2336, 'goal_height': 2.44}, 'active_goal': {}, 'game_id': '', 'half': 0, 'segment_id': '', 'td': Empty DataFrame
Columns: []
Index: [], 'td_frame': {}, 'ball_data': Empty DataFrame
Columns: []
Index: [], 'home_id': '', 'away_id': '', 'home_team_name': '', 'away_team_name': '', 'home_goalkeeper_ids': [], 'away_goalkeeper_ids': [], 'goalkeeper_ids': [], 'players_df': Empty DataFrame
Columns: []
Index: [], 'pitch_length': 105, 'pitch_width': 68, 'max_halves': 4, 'pz_setpiece': 3, 'player_in_pitch': {'sideline': 1, 'endline': 0.5, 'goal': 0.1}, 'ball_in_pitch': 1, 'in_area': 0.25, 'kickoff_player_out_pitch': 1, 'on_target_thr

# Run auto-eventing

In [7]:
tstart = time()

data = dict()

times_list = []
for i,game_id in enumerate(list_of_games,1):    
    
    ####### METADATA ####### 
    params = copy(params_base)
    t0 = time()
    players_df = parse_lineups(metadata_path, game_id)
    game_params = load_game_params(game_id, players_df)
    params.update(game_params)    

    if verbose:
        print(f"{params['home_team_name']} vs. {params['away_team_name']} -- ({i}/{len(list_of_games)})")    
        
    ####### TRACKING DATA #######
    t1 = time()
    
    ### Parse tracking data downloaded from FDP  
    gameDate = str(players_df.iloc[0]['date']).replace('-','').split(' ')[0]
    folder_name = f"TRACKING_DATA_{gameDate}_{players_df.iloc[0]['home_acronym']}-{players_df.iloc[0]['away_acronym']}"
    path_to_data = tracking_raw_path+f'{folder_name}/'
    if os.path.exists(path_to_data+f'{game_id}.dat'):                          
        with open(path_to_data+f'{game_id}_metadata.json','r') as g:
            meta_data = json.loads(g.readlines()[0])  
        params['pitch_length'] = float(meta_data['PitchLongSide'])/100
        params['pitch_width'] = float(meta_data['PitchShortSide'])/100

        tracking_data = np.genfromtxt(path_to_data+f'{game_id}.dat',names=True,dtype=None,encoding='UTF-8',delimiter='\n')   
        halves_df = get_halves_tracab(params, meta_data)
    else:
        ### No tracking data found, moving on to the next segment
        if verbose:
            print("---------------------------------------------------------------------")
        continue
    
    for this_half in halves_df.itertuples():
        
        this_half = this_half._asdict()        
        params_this_half = copy(params)
        
        # Initialize output file
        this_segment = {}
        segment_id = f"{game_id}_{this_half['half']}"
        params_this_half['segment_id'] = 	segment_id
        params_this_half['half']	= int(this_half['half'])				

        # Parse tracking data
        formatted_tracking_dict, _ = parse_half_tracab(this_half, tracking_data, params_this_half)
        this_segment['params'] = prepare_tracking_data(formatted_tracking_dict, params_this_half)
        del formatted_tracking_dict
        
        if verbose:
            print(f'Tracking data parsed in {np.round((time()-t1),1)} seconds')
    
        ####### POSSESSION MODEL #######
        this_segment = possession_model(this_segment)

        ####### EVENT DETECTION MODEL #######
        this_segment = event_detection(this_segment)
    
    
        if verbose:
            print(f"Game half done in {np.round((time()-t0),1)} seconds")
            print("---------------------------------------------------------------------")

        ####### SAVE OUTPUT #######
        if save_data:
            this_segment['params'].pop('td')
            this_segment['params'].pop('db_from_sp')
            this_segment['params'].pop('td_frame')
            this_segment['params'].pop('ball_data')
            if not os.path.exists(output_path+this_run):
                os.makedirs(output_path+this_run)
            compressed_pickle(output_path+this_run+f'/{segment_id}',this_segment)
        data[segment_id] = this_segment


    if verbose:
        print("---------------------------------------------------------------------")
        print("---------------------------------------------------------------------")
        print("---------------------------------------------------------------------")



Qatar vs. Ecuador -- (1/64)
10000 at 2023-11-10 09:59:36.530710
20000 at 2023-11-10 09:59:37.243187
30000 at 2023-11-10 09:59:37.816867
40000 at 2023-11-10 09:59:38.510365
50000 at 2023-11-10 09:59:39.170396
60000 at 2023-11-10 09:59:39.700360
70000 at 2023-11-10 09:59:40.407584
80000 at 2023-11-10 09:59:40.671962
90000 at 2023-11-10 09:59:40.707028
100000 at 2023-11-10 09:59:40.740946
110000 at 2023-11-10 09:59:40.774780
120000 at 2023-11-10 09:59:40.808309
130000 at 2023-11-10 09:59:40.841952
140000 at 2023-11-10 09:59:40.875540
150000 at 2023-11-10 09:59:40.910308
160000 at 2023-11-10 09:59:40.946514
170000 at 2023-11-10 09:59:40.981797



KeyboardInterrupt

